In [ ]:
from dask.distributed import Client, LocalCluster, wait, progress
import gc
import os

import update_funcs

client = update_funcs.initialize_dask(w=4, t=1, mem=24, disk=20)

In [ ]:
ddf1_base, features_ddf_base, shape, chunksize = update_funcs.load_ddfs(
    '500M')
#     'small')

base_feature_name = 'feature1'
feature_name1 = 'feature10'
feature1 = (-1, 0, 1)
feature_name2 = 'feature20'
feature2 = (2, 3, -2)

In [ ]:
# %%timeit -r 1 -n 1

ddf1, features_ddf, chunksize = update_funcs.repartition(
    ddf1_base, features_ddf_base, 100)

import ctypes

def trim_memory() -> int:
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

client.run(trim_memory)

In [ ]:
# %%timeit -r 1 -n 1

# features_min = features_ddf.index.map_partitions(min).compute().to_numpy()
# features_max = features_ddf.index.map_partitions(max).compute().to_numpy()

In [ ]:
features_min = features_ddf.index.map_partitions(min).compute().to_numpy()
features_max = features_ddf.index.map_partitions(max).compute().to_numpy()

ddf1[feature_name1] = ddf1.map_partitions(
    update_funcs.update_part_no_minmax,
    update_funcs.Wrapper(features_ddf),
    features_min,
    features_max,
    shape,
    feature1,
    base_feature_name,
    chunksize,
    align_dataframes=False,
    meta=(None, int))
ddf1 = ddf1.persist()
progress(ddf1)